# `04_sample_descriptives.ipynb`

### What it does
- Loads the merged analysis dataset from the Research Drive (`final_merged_dataset_for_analysis.parquet`).
- Defines the **active sample** as participants (`uid`) who appear in at least one annotation row for:
  - `stellingen.misinformation`
  - `stellingen.sentiment`
  - `stellingen.toxic`
- Collapses the dataset to **one row per participant** (first non-missing value per `uid`) for a predefined set of baseline variables.
- Produces a publication-ready **LaTeX descriptives table** (English labels), including:
  - Continuous variables as **Mean (SD)**
  - Categorical variables as **count (percentage)** using non-missing denominators
- Uploads the LaTeX table to the Research Drive.

### Inputs
- Merged analysis dataset:
  - `data/final_merged_dataset_for_analysis.parquet`
- Configuration and utilities:
  - `config.PROJECT_ROOT`
  - `rd_utils` (WebDAV-enabled helpers)

### Output
- LaTeX table (Research Drive, under `output/tables/`):
  - `output/tables/active_participants_demographics_EN.tex`


In [1]:
import numpy as np
import pandas as pd
import config
import rd_utils as rd
from rd_utils import webdav_mkdirs, webdav_upload_bytes

# ---------------------------
# Paths (Research Drive)
# ---------------------------
PROJ = config.PROJECT_ROOT
DATA_DIR = f"{PROJ}/data"
TABLES_DIR = f"{PROJ}/output/tables"

INPUT_PARQUET = "final_merged_dataset_for_analysis.parquet"
OUTPUT_TEX_FILENAME = "active_participants_demographics_EN.tex"

# ---------------------------
# Read data (remote)
# ---------------------------
df = rd.read_parquet(f"{DATA_DIR}/{INPUT_PARQUET}")
print(f"[Load] merged dataset: {df.shape[0]:,} rows | {df.shape[1]:,} cols")

# ---------------------------
# Variables
# ---------------------------
stelling_vars = [
    "stellingen.misinformation",
    "stellingen.toxic",
    "stellingen.sentiment",
]

continuous_vars = [
    "Age", "dsc", "vote_likelihood_score",
    "ImportIssue_1_numeric", "KnowIssue_1_numeric", "AttitudeExtr2_combined"
]

categorical_vars = [
    "Gender", "Etniciteit_binair", "Edu_breed",
    "PolOrientation_cat", "Werk_binair"
]

# ---------------------------
# Helpers
# ---------------------------
def percent(n, d):
    return 0.0 if d == 0 else (100.0 * n / d)

def latex_escape(s):
    return (str(s)
            .replace("&", r"\&").replace("%", r"\%").replace("_", r"\_")
            .replace("#", r"\#").replace("$", r"\$").replace("{", r"\{")
            .replace("}", r"\}").replace("~", r"\textasciitilde{}")
            .replace("^", r"\textasciicircum{}"))

# ---------------------------
# Ensure coding (only if missing)
# ---------------------------
# Ethnicity binary
if "Etniciteit_binair" not in df.columns and "Etniciteit" in df.columns:
    df["Etniciteit_binair"] = df["Etniciteit"].apply(lambda x: 1 if str(x).strip() == "Nederlands" else 0)

# Education order
edu_order_nl = ["Basisonderwijs", "Voortgezet onderwijs", "Praktijkopleiding", "Hoger onderwijs"]
if "Edu_breed" in df.columns:
    df["Edu_breed"] = pd.Categorical(df["Edu_breed"], categories=edu_order_nl, ordered=True)

# Employment binary
if "Werk_binair" not in df.columns and "Werk" in df.columns:
    df["Werk_binair"] = df["Werk"].apply(
        lambda x: 1 if x in ["Werkend (betaalde werknemer)", "Werkend (zelfstandig ondernemer)"] else 0
    )

# ---------------------------
# Labels + translations
# ---------------------------
var_label_map = {
    "Age": "Age",
    "dsc": "Implicit prejudice toward Arab-Muslims",
    "vote_likelihood_score": "Ideological congruence",
    "ImportIssue_1_numeric": "Issue importance",
    "KnowIssue_1_numeric": "Issue knowledge",
    "AttitudeExtr2_combined": "Anti-immigrant attitude",
    "Gender": "Gender",
    "Etniciteit_binair": "Ethnicity",
    "Edu_breed": "Education",
    "PolOrientation_cat": "Political orientation",
    "Werk_binair": "Employment status",
}

def translate_gender(x):
    return {"Vrouw": "Female", "Man": "Male", "Anders": "Other"}.get(x, "Missing")

def translate_eth_bin(x):
    return {1: "Dutch", 0: "Non-Dutch"}.get(x, "Missing")

def translate_edu(x):
    return {
        "Basisonderwijs": "Primary education",
        "Voortgezet onderwijs": "Secondary education",
        "Praktijkopleiding": "Vocational training",
        "Hoger onderwijs": "Higher education",
    }.get(x, "Missing")

def translate_pol(x):
    return {"Midden": "Center", "Rechts": "Right", "Links": "Left"}.get(x, "Missing")

def translate_work(x):
    return {1: "Employed", 0: "Not employed"}.get(x, "Missing")

category_translators = {
    "Gender": translate_gender,
    "Etniciteit_binair": translate_eth_bin,
    "Edu_breed": translate_edu,
    "PolOrientation_cat": translate_pol,
    "Werk_binair": translate_work,
}

# ---------------------------
# Subset to "active" participants
# ---------------------------
for col in ["uid", "variable"]:
    if col not in df.columns:
        raise KeyError(f"Expected column '{col}' not found in merged dataset.")

active_uids = df.loc[df["variable"].isin(stelling_vars), "uid"].dropna().unique()
print(f"[Active] unique active uids: {len(active_uids):,}")

cols_needed = [c for c in (continuous_vars + categorical_vars) if c in df.columns]
missing_cols = [c for c in (continuous_vars + categorical_vars) if c not in df.columns]
if missing_cols:
    print(f"[Warn] missing columns (skipped): {missing_cols}")

active_demo = (
    df.loc[df["uid"].isin(active_uids), ["uid"] + cols_needed]
      .groupby("uid")[cols_needed]
      .first()
      .copy()
)
n_active = active_demo.shape[0]
print(f"[Active] one-row-per-uid table: {n_active:,} participants")

# ---------------------------
# Build LaTeX rows
# ---------------------------
rows = []
rows.append(r"\multicolumn{3}{l}{Participants: " + f"N={n_active}" + r"} \\")
rows.append(r"\addlinespace")

# Continuous
for v in continuous_vars:
    if v not in active_demo.columns:
        continue
    x = pd.to_numeric(active_demo[v], errors="coerce").dropna()
    if x.empty:
        rows.append(f"{latex_escape(var_label_map.get(v, v))} &  & \\\\")
        continue
    mean = x.mean()
    sd = x.std(ddof=1)
    rows.append(f"{latex_escape(var_label_map.get(v, v))} & {mean:.2f} ({sd:.2f}) & \\\\")
rows.append(r"\addlinespace")

# Categorical
for v in categorical_vars:
    if v not in active_demo.columns:
        continue
    series = active_demo[v]
    translated = series.map(category_translators.get(v, lambda z: z))
    counts = translated.value_counts(dropna=False)

    rows.append(f"{latex_escape(var_label_map.get(v, v))} &  & \\\\")
    denom = int(series.notna().sum())

    for level, n in counts.items():
        if level == "Missing" or pd.isna(level):
            continue
        n = int(n)
        pct = percent(n, denom)
        rows.append(r"\quad " + f"{latex_escape(level)} & {n} ({pct:.1f}\\%) & \\\\")
    rows.append(r"\addlinespace")

# ---------------------------
# Assemble LaTeX table (WITH your extended notes)
# ---------------------------
table_notes = (
    r"Values for scales are reported as Mean (SD). "
    r"Categorical variables are reported as count (percentage), using non-missing denominators. "
    r"Ethnicity (binary): 1 = Dutch, 0 = non-Dutch. "
    r"Education ordered as Primary $\to$ Secondary $\to$ Vocational $\to$ Higher. "
    r"Employment: 1 = employed (paid worker or self-employed), 0 = other. "
    r"Anti-immigrant attitudes was measured on a scale based on eight immigration attitude items adapted from "
    r"Azrout, van Spanje \& de Vreese (2011). "
    r"Issue importance and knowledge are single 5-point items "
    r"(1 = very unimportant / very little, 5 = very important / very much). "
    r"References: Azrout et al. (2011); Hameleers \& van der Meer (2023); Sap et al. (2021); "
    r"Babakov et al. (2021); van der Velden et al. (2023); Wojcieszak (2012); Zerback \& Kobilke (2022)."
)

latex_table = (
    r"\begin{table}[ht]" "\n"
    r"\centering" "\n"
    r"\caption{Participant characteristics}" "\n"
    r"\label{tab:active_participants_demographics}" "\n"
    r"\begin{tabular}{lcc}" "\n"
    r"\toprule" "\n"
    r"\textbf{Variable} & \textbf{Value} & \textbf{Notes} \\" "\n"
    r"\midrule" "\n"
    + "\n".join(rows) + "\n"
    r"\bottomrule" "\n"
    r"\end{tabular}" "\n"
    r"\begin{tablenotes}" "\n"
    r"\footnotesize " + table_notes + "\n"
    r"\end{tablenotes}" "\n"
    r"\end{table}"
)

# ---------------------------
# Upload to Research Drive
# ---------------------------
webdav_mkdirs(TABLES_DIR)
rel_path = f"{TABLES_DIR}/{OUTPUT_TEX_FILENAME}"
webdav_upload_bytes(rel_path, latex_table.encode("utf-8"), content_type="text/plain")

print(f"\nUploaded demographics table to Research Drive: {rel_path}")

[Load] merged dataset: 153,674 rows | 203 cols
[Active] unique active uids: 1,248
[Active] one-row-per-uid table: 1,248 participants

Uploaded demographics table to Research Drive: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/active_participants_demographics_EN.tex


In [2]:
# How many unique tweets each participant annotated
tweets_per_part = df.groupby("uid")["tweet_id"].nunique()

print("\nTweets annotated per participant:")
print(tweets_per_part.describe())


Tweets annotated per participant:
count    1358.000000
mean       25.634757
std         9.534904
min         0.000000
25%        28.000000
50%        30.000000
75%        30.000000
max        30.000000
Name: tweet_id, dtype: float64


In [3]:
# How many batches per participant
batches_per_part = df.groupby("uid")["jobset"].nunique()

print("\nBatches annotated per participant:")
print(batches_per_part.describe())


Batches annotated per participant:
count    1358.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: jobset, dtype: float64
